In [23]:
import argparse
import os
import numpy as np
import pandas as pd
import random
import sys
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm

from rdkit import rdBase, Chem
rdBase.DisableLog('rdApp.error')

from models import RNN, OneHotRNN, EarlyStopping
from datasets import SmilesDataset, SmilesCollate
from functions import decrease_learning_rate, print_update, track_loss, \
    sample_smiles, write_smiles
#from util.SmilesEnumerator import SmilesEnumerator
from functions import clean_mol,clean_mols,read_smiles, write_smiles

import dill

In [3]:
import os
import numpy as np
import pandas as pd
import random
from tqdm import tqdm

from rdkit import rdBase, Chem
rdBase.DisableLog('rdApp.error')

In [6]:
smile = read_smiles("D:\OneDrive\data for all\Pubchem dataset.txt")
len(smile)

10000000

In [7]:
smile[0]

'CN(c1ccccc1)c1ccccc1C(=O)NCC1(O)CCOCC1'

In [8]:
def get_total_charge(mol):
    return sum([atom.GetFormalCharge() for atom in mol.GetAtoms()])

In [ ]:
negatively_charged_smiles = []
positively_charged_smiles = []
for smiles in smile:
    try:
        mol = Chem.MolFromSmiles(smiles)
        if get_total_charge(mol) >0:
            positively_charged_smiles.append(smiles)
        if get_total_charge(mol) <0:
            negatively_charged_smiles.append(smiles)
    except:
        pass

In [6]:
len(positively_charged_smiles),len(negatively_charged_smiles)

(3037037, 873783)

In [7]:
positively_charged_smiles.extend(negatively_charged_smiles)

In [8]:
len(positively_charged_smiles)

3910820

In [9]:
with open('charged_smiles.txt', 'w') as file:
    for smiles in positively_charged_smiles:
        file.write(smiles + '\n')

In [10]:
##Augmentation
smiles = read_smiles('charged_smiles.txt')
smiles = np.asarray(smiles)
sme = SmilesEnumerator(canonical=False, enum=True)
summary = pd.DataFrame()
enum = []
max_tries = 200 ## randomized SMILES to generate for each input structure
for sm_idx, sm in enumerate(tqdm(smiles)):
    tries = []
    for try_idx in range(max_tries):
        this_try = sme.randomize_smiles(sm)
        tries.append(this_try)
        tries = [rnd for rnd in np.unique(tries)]
        if len(tries) > 10:
            tries = tries[:10]
            break
    enum.extend(tries)
print(len(enum))
write_smiles(enum,'smile_aug.txt')

100%|██████████| 3910820/3910820 [4:32:26<00:00, 239.24it/s]  


39106891


In [ ]:
#dataset = SmilesDataset(smiles_file='smile_aug.txt',training_split=0.9)

In [2]:
#smile = read_smiles('smile_aug.txt')

39106891

In [2]:
dataset = SmilesDataset(smiles_file='charged_smiles_after_clean.txt',training_split=0.9)

In [3]:
with open('data/dataset', 'wb') as file:
    dill.dump(dataset, file)

In [5]:
dataset = dill.load(open('data/dataset', "rb"))

In [6]:
len(dataset)

3515916

In [7]:
loader = DataLoader(dataset,
                    batch_size=64,
                    shuffle=True,
                    drop_last=False,
                    collate_fn=SmilesCollate(dataset.vocabulary))

In [8]:
#设定随机种子，便于复现
torch.manual_seed(100)
random.seed(100)
np.random.seed(100)
#判断CUDA是否可用
if torch.cuda.is_available():
    print("using cuda")
    torch.cuda.manual_seed_all(100)

using cuda


In [9]:
model = RNN(vocabulary=dataset.vocabulary,
                rnn_type='GRU',
                embedding_size=128,
                hidden_size=512,
                n_layers=3,
                dropout=0.2,
                bidirectional=False,
                nonlinearity='relu')
optimizer = optim.Adam(model.parameters(),
                       betas=(0.9, 0.999), ## default
                       eps=1e-08, ## default
                       lr=0.0001)

In [7]:
class early_stopping:
    def __init__(self, patience):
        self.patience = patience
        self.counter = 0
        self.early_stop = False
        self.best_score = None
        self.loss_min = np.Inf
    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(model)
        elif score < self.best_score:
            self.counter += 1
            if self.counter > self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.counter = 0
            self.loss_min = val_loss
            self.save_checkpoint(model)
    def save_checkpoint(self, model):
        torch.save(model.state_dict(), 'data/best.pth')
early_stop = early_stopping(patience=5)

In [8]:
def Val():
    model.eval()    #将模型设置为评估模式
    test_loss = []
    with torch.no_grad():   #关闭torch的自动求导，降低内存占用，加快运算速度
        for j in range(2746):
            val_batch, val_length = validation[j*128:(j+1)*128, :], lengths[j*128:(j+1)*128]
            log_p = model.loss(val_batch, val_length)
            loss_val = log_p.mean()
            test_loss.append(loss_val.cpu().numpy())
    validation_loss = np.mean(test_loss)
    return validation_loss

In [9]:
validation, lengths = dataset.get_validation(351591)

In [10]:
validation.shape

torch.Size([351591, 922])

In [12]:
Val()

1.2606324

In [13]:
sched_filename = "training_schedule-" + str(0 + 1) + ".csv"
sched_file = os.path.join('data/', sched_filename)
counter = 0
for epoch in range(200):
    model.train()
    for batch_idx, batch in tqdm(enumerate(loader), total=len(loader)):
        batch, leng = batch
        counter += 1
        loss = model.loss(batch, leng)
        loss = loss.mean()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    #decrease_learning_rate(optimizer,multiplier=0.5)
    validation_loss = Val()
    early_stop(validation_loss, model)
    print_update(model, dataset, epoch, epoch, loss.item(), 1000)
    #track_loss(sched_file, model, dataset, epoch,counter, loss.item(), validation_loss)
    #sample_smiles('data/', 0, model,1000, epoch, counter)
    if early_stop.early_stop:
        break

100%|██████████| 54937/54937 [4:20:23<00:00,  3.52it/s]  


**************************************************
epoch   0 -- step   0 -- loss:  0.75 -- validation loss:  0.69
CCOC1CC(NC(=O)CC(C(=O)OC)C(CCOCC)=C(C)OC)c(C(=O)[O-])c2ccccc21
[NH3+]CCCc1ccc(Nc2ncc(F)c(C(F)(F)F)n2)cc1
COc1cc(Cl)cc(C)c1C(=O)NC(C)C1CCC[NH2+]1
Cc1nnc(SCC2=CC[NH2+]CC2)cc1N(C)C
Cc1ccc(S(=O)(=O)N2CCN(C(=O)C3CCC[NH+](CC(=O)N4CCCCC4CCO)C3)CC2)cc1
89.3% valid SMILES



100%|██████████| 54937/54937 [4:41:10<00:00,  3.26it/s]   


**************************************************
epoch   1 -- step   1 -- loss:  0.80 -- validation loss:  0.69
CC1=C(C)CC(COc2ccc3c(c2)C2(C)C4(C)CC(c5cccnc5)(C(=O)[O-])=CC(S(C)(=O)=O)=CCCC4(C(=O)[O-])CCC23C)C1
Cc1ccc(C(CNC(=O)CS(C)(=O)=O)[NH+]2CCOCC2)cc1
Cc1cc(C(C)C)ccc1OCC(=O)NC(Cc1ccccc1)C(=O)[O-]
CC(=O)Oc1ccc(SC2CCCCC2)c(C(=O)[O-])c1
CC(C)(C)Oc1ccc2nc(N3CC[NH2+]CC3)nc(N3CCCCC3)c2c1
92.9% valid SMILES



100%|██████████| 54937/54937 [4:14:22<00:00,  3.60it/s]  


...

**************************************************
epoch  13 -- step  13 -- loss:  0.62 -- validation loss:  0.64
CCCNC(=O)C(C)[NH2+]Cc1cc(C)c(C)o1
Cc1cc(Cl)ccc1NC(=O)C(C)[NH+](C)Cc1cc(Br)cs1
C[NH+](Cc1cnn(-c2ccccc2)c1)CC(O)CC(C)(C)C
CCc1ccc(C(=O)C[NH+]2CCOC(C(N)=O)C2)cc1
C[NH2+]C(C)c1csc(CCOc2ccccc2Cl)n1
95.1% valid SMILES



100%|██████████| 54937/54937 [1:51:22<00:00,  8.22it/s]  


**************************************************
epoch  14 -- step  14 -- loss:  0.70 -- validation loss:  0.64
CC(C)(C)[Si](C)(C)OC1CC(CC(=O)[O-])C1
CCC[NH2+]CCC[NH2+]Cc1ccc(OCC(=O)OCC)cc1
Cc1nn(C)cc1C([NH3+])c1cccc2c1CCCC2
CCNc1ccc([N+](=O)[O-])c(NCC([NH3+])C2CC2)n1
CCCC1NCc2ccc(NC3CC[NH2+]CC3)cc21
94.3% valid SMILES



  3%|▎         | 1843/54937 [03:44<1:48:00,  8.19it/s]


KeyboardInterrupt: 

In [14]:
##loaded well trained model
model_file =  'data/best.pth'
model.load_state_dict(torch.load(model_file))
model.eval() ## enable evaluation modes
smiles_file = 'data/sample_smile.smi'
sampled_smiles = []
while len(sampled_smiles) < 10000:
    sampled_smiles.extend(model.sample(256, return_smiles=True))
# write sampled SMILES
write_smiles(sampled_smiles, smiles_file)
#sample_smiles(dir_data + '/hh', 0, model, 100000, 1, 0)

In [15]:
from rdkit import Chem
from functions import clean_mols, read_smiles
gen_smiles = read_smiles(smiles_file)
gen_mols = [mol for mol in clean_mols(gen_smiles,selfies=False, deepsmiles=False, stereochem=False) if mol]
gen_canonical = [Chem.MolToSmiles(mol) for mol in gen_mols]

100%|██████████| 10240/10240 [00:01<00:00, 6194.44it/s]


In [16]:
##caculate valid percent
pct_valid = len(gen_mols) / len(gen_smiles)
pct_valid

0.9486328125

In [ ]:
###store the unique values
g_s = pd.DataFrame(columns=['ca-smile'], data = np.unique(gen_canonical))
g_s.head()

,ca-smile
0,BrC1(c2ccc[nH]2)CC2CCc3cccc[n+]3C21
1,BrC1=CC(Br)=C2SCC(=C1)c1nn[n+](-c3ccc(Br)cc3)n12
2,BrCCCCCCC[S+]1CCCC1
3,BrCCC[P+](c1ccccc1)(c1ccccc1)c1ccsc1
4,Brc1cc2[nH]c3ccccc3c2[n+](Cc2c3ccccc3cc3ccccc2...


In [19]:
## save the model
import pickle
model_file =  'data/best.pth'
model.load_state_dict(torch.load(model_file))
with open('data/generate_model.model', 'wb') as f:
    pickle.dump(model, f)

In [1]:
import pickle
model = pickle.load(open('data/generate_model.model' , "rb"))

C:\Users\Lenovo\anaconda3\envs\NatureFilter-GP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
### checking the saved model
smiles_file = 'data/sample_smile.smi'
sampled_smiles = []
while len(sampled_smiles) < 10000:
    sampled_smiles.extend(model.sample(256, return_smiles=True))
# write sampled SMILES
write_smiles(sampled_smiles, smiles_file)
#sample_smiles(dir_data + '/hh', 0, model, 100000, 1, 0)

C:\Users\Lenovo\anaconda3\envs\NatureFilter-GP\lib\site-packages\torch\nn\modules\rnn.py:950: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at  C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\cudnn\RNN.cpp:968.)
  result = _VF.gru(input, hx, self._flat_weights, self.bias, self.num_layers,


In [4]:
from rdkit import Chem
from functions import clean_mols, read_smiles
gen_smiles = read_smiles(smiles_file)
gen_mols = [mol for mol in clean_mols(gen_smiles,selfies=False, deepsmiles=False, stereochem=False) if mol]
gen_canonical = [Chem.MolToSmiles(mol) for mol in gen_mols]

100%|██████████| 10240/10240 [00:01<00:00, 6645.67it/s]


In [5]:
##caculate valid percent，目前比较稳定在90%以上
pct_valid = len(gen_mols) / len(gen_smiles)
pct_valid

0.91162109375

In [46]:
cations = []
anions = []
for smile in gen_canonical:
    mol = Chem.MolFromSmiles(smile)
    if Chem.GetFormalCharge(mol) > 0:
        cations.append(Chem.MolToSmiles(mol))
    elif Chem.GetFormalCharge(mol) < 0:
        anions.append(Chem.MolToSmiles(mol))

df_cations = pd.DataFrame({'new_cations': cations})
df_anions = pd.DataFrame({'new_anions': anions})
df_combined = pd.concat([df_cations, df_anions], axis=1)
df_combined.to_csv('data/generate_ions.csv', index=False)

In [35]:
#seperate cations and anions, no need to use
record_frags = []
i = 0
for smile in gen_smiles:
    try:
        mol = clean_mol(smile)
    except:
        continue
    frags = Chem.GetMolFrags(mol, asMols=True)
    if len(frags) != 2:
        continue
    
    charges = [Chem.GetFormalCharge(frag) for frag in frags]
    smiles_frags = [Chem.MolToSmiles(frag) for frag in frags]
    
    if charges[0] > 0:
        cation = smiles_frags[0]
        anion = smiles_frags[1]
    else:
        cation = smiles_frags[1]
        anion = smiles_frags[0]
    
    record_frags.append({'original_smiles': smile,'cation': cation,'anion': anion})
gen_ions = pd.DataFrame(record_frags)
print(gen_ions.head())

Empty DataFrame
Columns: []
Index: []
